# Tokenization

* Our model input will be text-based input in sentence structures. So, we need to convert text to numeric values in order to utilize them for the training of the model.
* Conversion from text input to a numerical value is called word or  sub-word tokenization.
* Before diving into tokenization of the words, we need to chunk sentences into smaller pieces which can be words, sub-words or even characters.
* Once we have implemented the sentence chunking and used the tokens in training process, we need to convert tokens to original characters so the user will be able to understand the output from LLMs.
* To that attempt, we can build a class in python, let's call it `Tokenizer`, and the class can involve at least two methods: one of them is `encode` and the other is `decode` where the encode method takes the text input and converts it to a list of numeric tokens and the decode method will behave vice versa. We can also add some helper methods in Tokenizer.

Let's start reading text file from data folder and try to implement a basic tokenizer before diving into the class definition.

In [24]:
from textwrap import TextWrapper
from pathlib import Path


text_wrapper = TextWrapper(width=100)
f_path = Path("../data/the-verdict.txt")
with open(f_path, "r", encoding="utf-8") as file:
    raw_data = file.read()

print("Type of raw data: ", type(raw_data))
print("Number of characters: ", len(raw_data))
text_wrapper.wrap(raw_data[:500])

Type of raw data:  <class 'str'>
Number of characters:  20479


['I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no',
 'great surprise to me to hear that, in the height of his glory, he had dropped his painting, married',
 'a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would',
 'have been Rome or Florence.)  "The height of his glory"--that was what the women called it. I can',
 'hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course',
 "it'"]

Now, let's write a regular expression to able to separate the words and special characters using a test sentence.

In [37]:
import re

text = "Hello, world. Is this-- a test?"
result = re.split(r'(\s|[.,?]|--)', text)
result_without_space = [item.strip() for item in result if item.strip()]

result_without_space

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']

We can check the unique characters in the original text to add them in the special character list in regex definition.

In [63]:
# ^a-zA-Z0-9 -> matches with non-alphanumeric characters, a-zA-Z0-9 -> matches with alphanumeric characters
result = re.sub(r'([a-zA-Z0-9]|\s)', '', raw_data)
print("All characters: ", result)

unique_result_list = sorted(set(result))
print("All unique characters: ", unique_result_list)

# replace "-" with "--"
unique_result_list.remove("-")
unique_result_list.append("--")
print("All unique characters ('-' is replaced with '--'): ", unique_result_list)

unique_result_text = "".join(unique_result_list)
print("All unique characters joined: ", unique_result_text)

All characters:  ----,,,,.(.)""--..----."'';',.--.",.',__...,,'"-",:""?!--'.!--.,.?.,,,,""--(')'.----,,.,"".,',.,.--.".".----'.----';.,,,,""--.--!.--.;,,',.-;.',,."":.__----.,,:,-.""()..;.,,;'',-,.,,;-."',"-,,,;.,,:"."!:.,,.,,--?--,,.--.--.",''--,",.,.,,--,,.','.--__-".".,-."__?".-.",'__,;,".-,_-_,-."?'.".'."',.';'----."--'?-.:",.",,'.",',",;,-.,,,.'!.-,__,-,:".-,'."----'!--__-,.;,-,--,,,..,--,,--.'","--,,,,,,-'.,,""""--."',,"..","--"',.""?".,,,-,--.,.".,",.,.",,";,:".":-,-,-,---'.,:",'."--.,--,,--,,,:"!":"!",,.",",.:"";--,--,.'."'?",.","."---?",.",--'.".,;,---.",!".--,."--!".;,."!--.,?":"..""--'..""'--.....""??",:"--',,..--,!--.__."",--.__?"".,--.'-.',,.!'..""?--".",,--."."?""--.--!",."'--'.;'--.'',;."."',".,.."'--'.",-.",'--'!",,-.":--.",."?--'.....'.,,__--,,,.,__--!,--,.",--,,'!--..__--,,!.".,.-,,,--.,,..",:'.'---.:__,?--.",,--,..?,!,.,.'--';.,.!",.--,.!..-....",;.,,.,,.?'--....",,'.,'--__.,,--....,--.',,,?--;,''.',,--,,,:''?'",,.'--.,,,,',:''--''?"__--,'.,..'__--.',.--'!.--''!'--'.

Let's apply final special character list in regex for another test case.

In [69]:
text = """
"Hello, world. Is this-- a real test?", said 'Emanuel' and he roared (like a lion!). 
"""
result = re.split(r'(\s|[!"\'(),.:;?_]|--)', text)
result_without_space = [item.strip() for item in result if item.strip()]

print(result_without_space)

['"', 'Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'real', 'test', '?', '"', ',', 'said', "'", 'Emanuel', "'", 'and', 'he', 'roared', '(', 'like', 'a', 'lion', '!', ')', '.']


Apply final special character list in regex for raw data

In [74]:
result = re.split(r'(\s|[!"\'(),.:;?_]|--)', raw_data)
result_without_space = [item.strip() for item in result if item.strip()]

print(result_without_space[:40])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his']


Create token_id to token and token to token_id dictionaries

In [84]:
id_to_token_dict = {k: v for k, v in enumerate(sorted(set(result_without_space)))}

# print first 50 tokens from the dictionary
for id, item in id_to_token_dict.items():
    print(f"({id}, {item})")
    if id == 50:
        break

len(id_to_token_dict)

(0, !)
(1, ")
(2, ')
(3, ()
(4, ))
(5, ,)
(6, --)
(7, .)
(8, :)
(9, ;)
(10, ?)
(11, A)
(12, Ah)
(13, Among)
(14, And)
(15, Are)
(16, Arrt)
(17, As)
(18, At)
(19, Be)
(20, Begin)
(21, Burlington)
(22, But)
(23, By)
(24, Carlo)
(25, Chicago)
(26, Claude)
(27, Come)
(28, Croft)
(29, Destroyed)
(30, Devonshire)
(31, Don)
(32, Dubarry)
(33, Emperors)
(34, Florence)
(35, For)
(36, Gallery)
(37, Gideon)
(38, Gisburn)
(39, Gisburns)
(40, Grafton)
(41, Greek)
(42, Grindle)
(43, Grindles)
(44, HAD)
(45, Had)
(46, Hang)
(47, Has)
(48, He)
(49, Her)
(50, Hermia)


1130

In [85]:
token_to_id_dict = {v: k for k, v in enumerate(sorted(set(result_without_space)))}

# print first 50 tokens from the dictionary
for item, id in token_to_id_dict.items():
    print(f"({item}, {id})")
    if id == 50:
        break

len(token_to_id_dict)

(!, 0)
(", 1)
(', 2)
((, 3)
(), 4)
(,, 5)
(--, 6)
(., 7)
(:, 8)
(;, 9)
(?, 10)
(A, 11)
(Ah, 12)
(Among, 13)
(And, 14)
(Are, 15)
(Arrt, 16)
(As, 17)
(At, 18)
(Be, 19)
(Begin, 20)
(Burlington, 21)
(But, 22)
(By, 23)
(Carlo, 24)
(Chicago, 25)
(Claude, 26)
(Come, 27)
(Croft, 28)
(Destroyed, 29)
(Devonshire, 30)
(Don, 31)
(Dubarry, 32)
(Emperors, 33)
(Florence, 34)
(For, 35)
(Gallery, 36)
(Gideon, 37)
(Gisburn, 38)
(Gisburns, 39)
(Grafton, 40)
(Greek, 41)
(Grindle, 42)
(Grindles, 43)
(HAD, 44)
(Had, 45)
(Hang, 46)
(Has, 47)
(He, 48)
(Her, 49)
(Hermia, 50)


1130

Let's define the tokenizer class based on the previous implementations.

In [ ]:
class TokenizerV1:

    def __init__(self):
        pass

    def encode(self):
        pass

    def decode(self):
        pass